# RunPod H100：PoC B 冻结语义原型分类器

请把 Notebook 复制到仓库外的 `/workspace/runpod_poc_b_training.ipynb` 后执行，以免 Jupyter 输出污染 Git。流程为：环境安装、数据重建、32 类 smoke、1000 类完整训练、五个 milestone 评测、dry-run，最后人工发布。

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

HF_REPO_ID = 'hxgdzyuyi/qwen3-8b-steam-entity-linking-poc-b'
PUBLISH_PUBLIC = False
candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace/qwen-steam-entity-linking')]
PROJECT_DIR = next((p.resolve() for p in candidates if (p / 'poc_b/configs/qwen3_8b_frozen_prototype.yaml').is_file()), None)
if PROJECT_DIR is None:
    raise RuntimeError('未找到项目，请先克隆仓库到 /workspace。')
POC_DIR = PROJECT_DIR / 'poc_b'
CONFIG = POC_DIR / 'configs/qwen3_8b_frozen_prototype.yaml'
SMOKE_RUN_DIR = POC_DIR / 'outputs/runpod-smoke'
FULL_RUN_DIR = POC_DIR / 'outputs/runpod-full'
os.environ.setdefault('HF_HOME', '/workspace/.cache/huggingface')

def run(command: list[str]) -> None:
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, cwd=PROJECT_DIR, env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

print('PROJECT_DIR =', PROJECT_DIR)
print('HF_REPO_ID =', HF_REPO_ID)
print('PUBLISH_PUBLIC =', PUBLISH_PUBLIC)

## 1. 安装依赖并重建固定数据


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-r', str(POC_DIR / 'requirements-cloud.txt'), 'hf_transfer>=0.1.9,<1'])
run([sys.executable, str(POC_DIR / 'scripts/build_training_data.py')])
run([sys.executable, '-m', 'py_compile', *map(str, sorted((POC_DIR / 'scripts').glob('*.py')))])

## 2. 独立 32 类 smoke（不可发布）


In [ ]:
run([sys.executable, str(POC_DIR / 'scripts/train.py'), '--config', str(CONFIG), '--mode', 'smoke', '--run-dir', str(SMOKE_RUN_DIR)])
run([sys.executable, str(POC_DIR / 'scripts/evaluate.py'), '--run-dir', str(SMOKE_RUN_DIR), '--all-milestones'])

## 3. 完整 1000 类训练（支持从最新 resume 恢复）


In [ ]:
resume_dir = FULL_RUN_DIR / 'resume'
command = [sys.executable, str(POC_DIR / 'scripts/train.py'), '--config', str(CONFIG), '--mode', 'full', '--run-dir', str(FULL_RUN_DIR)]
if (resume_dir / 'training_state.pt').is_file():
    command.extend(['--resume-from', str(resume_dir)])
run(command)

## 4. 零训练原型 + 五个 milestone + PoC A 对比


In [ ]:
run([sys.executable, str(POC_DIR / 'scripts/evaluate.py'), '--run-dir', str(FULL_RUN_DIR), '--all-milestones'])

## 5. 发布前 dry-run 与人工公开

`--public` 会先转 private、上传、下载回读并重新评测，全部一致后才公开。请通过 RunPod Secret 注入 `HF_TOKEN`。


In [ ]:
run([sys.executable, str(POC_DIR / 'scripts/publish_hf.py'), '--run-dir', str(FULL_RUN_DIR), '--repo-id', HF_REPO_ID, '--dry-run'])

In [ ]:
if PUBLISH_PUBLIC:
    if not os.environ.get('HF_TOKEN'):
        raise RuntimeError('请先通过 RunPod Secret 注入 HF_TOKEN。')
    run([sys.executable, str(POC_DIR / 'scripts/publish_hf.py'), '--run-dir', str(FULL_RUN_DIR), '--repo-id', HF_REPO_ID, '--public'])
else:
    print('PUBLISH_PUBLIC=False：未执行远端上传。')